# SQL Gym — 01: Aggregations

Practice: `GROUP BY`, `HAVING`, conditional aggregation, date bucketing, and multi-table aggregation.
Write your SQL in the `%%solution N` cell and run it — results preview inline. Then run the check cell to validate.

**Tables:** `users`, `accounts`, `merchants`, `transactions`

In [34]:
from pathlib import Path
import sys

_cwd = Path.cwd()
_candidates = [_cwd / "sql", _cwd, _cwd.parent, _cwd.parent / "sql",
               _cwd.parent.parent, _cwd.parent.parent / "sql"]
_sql_dir = next((p for p in _candidates if (p / "utils" / "__init__.py").exists()), None)
if _sql_dir is None:
    raise RuntimeError(
        "Cannot locate sql/utils. Run: uv run jupyter lab from the project root."
    )
if str(_sql_dir) not in sys.path:
    sys.path.insert(0, str(_sql_dir))

DATA_DIR = _sql_dir / "data"

from utils import get_conn, check, register_sql_magic
from utils.checks.aggregations import Checker

conn = get_conn(DATA_DIR)
checker = Checker(conn)
register_sql_magic()
print("Ready. Tables: users, merchants, accounts, transactions")

Ready. Tables: users, merchants, accounts, transactions


In [35]:
for table in ["users", "merchants", "accounts", "transactions"]:
    print(f"\n{'─'*50}\n  {table}\n{'─'*50}")
    display(conn.execute(f"SELECT * FROM {table} LIMIT 3").df())
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  ({n:,} rows total)")


──────────────────────────────────────────────────
  users
──────────────────────────────────────────────────


,user_id,name,email,country,tier,kyc_verified,created_date
0,1,Carlos Clark,user1@example.com,IN,basic,True,2021-02-09
1,2,Tara Hernandez,user2@example.com,SG,basic,False,2022-12-11
2,3,Marcus Robinson,user3@example.com,AU,basic,True,2020-02-15


  (500 rows total)

──────────────────────────────────────────────────
  merchants
──────────────────────────────────────────────────


,merchant_id,name,mcc_category,city,country
0,1,DailyBasket Groceries 1,groceries,Austin,UK
1,2,GreenLeaf Groceries 2,groceries,New York,IN
2,3,DailyBasket Groceries 3,groceries,Chicago,UK


  (200 rows total)

──────────────────────────────────────────────────
  accounts
──────────────────────────────────────────────────


,account_id,user_id,account_type,opened_date,balance,status
0,1,1,checking,2021-02-21,13714.20,active
1,2,2,savings,2023-01-21,3240.81,active
2,3,3,credit,2020-07-19,5271.51,active


  (600 rows total)

──────────────────────────────────────────────────
  transactions
──────────────────────────────────────────────────


,txn_id,account_id,merchant_id,amount,txn_type,txn_date,status
0,1,571,136,18.77,debit,2024-09-21,completed
1,2,21,124,17.97,debit,2024-12-19,completed
2,3,564,165,198.60,debit,2022-10-09,completed


  (20,000 rows total)


## Problem 1: Total Spending by MCC Category

Compute total debit spending across all completed transactions for each merchant category.

<details>
<summary>Hint</summary>

Join `transactions` with `merchants` on `merchant_id`. Filter `txn_type = 'debit'` and `status = 'completed'`. Group by `mcc_category` and sum `amount`. Sort descending.

</details>

| Column | Type | Notes |
|--------|------|-------|
| mcc_category | string | merchant category |
| total_spent | double | 2 decimal places, sorted DESC |

Expected: **8 rows** (one per MCC category).

In [36]:
%%solution 1

SELECT
mcc_category,
ROUND(SUM(amount), 2) total_spent
FROM transactions txn
JOIN merchants mr ON txn.merchant_id = mr.merchant_id 
WHERE txn_type = 'debit' AND status = 'completed'
GROUP BY mcc_category
ORDER BY total_spent DESC

,mcc_category,total_spent
0,utilities,314650.45
1,groceries,310219.95
2,retail,287459.54
3,travel,269339.87
4,fuel,238038.97
5,healthcare,232638.51
6,entertainment,206395.46
7,restaurants,182813.57


In [37]:
checker.p1(solution_1)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 2: Top 10 Merchants by Transaction Count

Find the 10 most frequently used merchants across all transactions (any status), showing their name, category, transaction count, and total transaction amount.

<details>
<summary>Hint</summary>

Join `transactions` with `merchants`. No status filter needed (count all). Group by `merchant_id`, `name`, `mcc_category`. ORDER BY `txn_count DESC`, LIMIT 10.

</details>

| Column | Type | Notes |
|--------|------|-------|
| merchant_id | integer | |
| name | string | merchant name |
| mcc_category | string | |
| txn_count | bigint | |
| total_amount | double | 2 decimal places |

Expected: **10 rows**.

In [38]:
%%solution 2

SELECT
txn.merchant_id,
mr.name,
mr.mcc_category,
COUNT(*) txn_count,
ROUND(SUM(txn.amount), 2) total_amount
FROM transactions txn
JOIN merchants mr
ON txn.merchant_id = mr.merchant_id
GROUP BY
txn.merchant_id,
mr.name,
mr.mcc_category
ORDER BY
txn_count DESC
LIMIT 10


,merchant_id,name,mcc_category,txn_count,total_amount
0,22,DailyBasket Groceries 22,groceries,286,51430.07
1,103,PowerGrid Utilities 3,utilities,278,55894.54
2,175,PharmaDirect Healthcare 25,healthcare,264,39850.56
3,187,GasPro Fuel 12,fuel,263,30686.79
4,146,MegaShop Retail 21,retail,260,39470.35
5,111,PowerGrid Utilities 11,utilities,258,37897.56
6,67,RoamEasy Travel 17,travel,257,45563.45
7,5,FreshMart Groceries 5,groceries,254,48445.40
8,90,CinePlex Entertainment 15,entertainment,252,35873.62
9,10,DailyBasket Groceries 10,groceries,249,44594.73


In [39]:
checker.p2(solution_2)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 3: Monthly Transactions by Account Type

For completed transactions, show the monthly transaction count and total amount broken down by account type. Use `DATE_TRUNC` for the monthly bucket.

<details>
<summary>Hint</summary>

Join `transactions` with `accounts` on `account_id`. Filter `status = 'completed'`. Group by `DATE_TRUNC('month', txn_date)::DATE` and `account_type`. In DuckDB and PostgreSQL: `DATE_TRUNC('month', col)`. Snowflake uses the same syntax. BigQuery reverses args: `DATE_TRUNC(col, MONTH)`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| month | date | monthly bucket |
| account_type | string | checking / savings / credit |
| txn_count | bigint | |
| total_amount | double | 2 decimal places |

Expected: rows ordered by month ASC, account_type ASC.

In [53]:
%%solution 3

SELECT
DATE_TRUNC('month', txn_date) AS month,
account_type,
COUNT(*) AS txn_count,
ROUND(SUM(amount), 2) AS total_amount
FROM transactions txn
JOIN accounts acc
ON txn.account_id = acc.account_id
WHERE txn.status = 'completed'
GROUP BY 1, 2
ORDER BY 1, 2 ASC


,month,account_type,txn_count,total_amount
0,2022-01-01,checking,240,33244.90
1,2022-01-01,credit,85,9386.81
2,2022-01-01,savings,127,20023.40
3,2022-02-01,checking,229,36108.30
4,2022-02-01,credit,68,16343.55
5,2022-02-01,savings,114,19739.11
6,2022-03-01,checking,241,35153.83
7,2022-03-01,credit,56,7213.50
8,2022-03-01,savings,135,29924.48
9,2022-04-01,checking,257,32624.53


In [54]:
checker.p3(solution_3)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 4: High-Value Users (Avg Debit > $500)

Find users whose average completed debit transaction amount exceeds $500. Return their user info, average transaction amount, and how many qualifying transactions they have.

<details>
<summary>Hint</summary>

Three-table join: `transactions → accounts → users`. Filter `txn_type = 'debit'` and `status = 'completed'`. Use `HAVING AVG(amount) > 500`. Note: HAVING filters *after* grouping; WHERE filters before.

</details>

| Column | Type | Notes |
|--------|------|-------|
| user_id | integer | |
| name | string | |
| country | string | |
| tier | string | |
| avg_txn_amount | double | 2 decimal places, sorted DESC |
| txn_count | bigint | |

Expected: variable number of rows, sorted by avg_txn_amount DESC.

In [70]:
%%solution 4

SELECT
acc.user_id,
users.name,
users.country,
users.tier,
AVG(txn.amount) AS avg_txn_amount,
COUNT(*) AS txn_count
FROM transactions txn
JOIN accounts acc
ON txn.account_id = acc.account_id
JOIN users
ON acc.user_id = users.user_id
WHERE txn.status = 'completed' AND txn.txn_type = 'debit'
GROUP BY 1, 2, 3, 4
HAVING avg_txn_amount > 500
ORDER BY avg_txn_amount DESC

,user_id,name,country,tier,avg_txn_amount,txn_count
0,205,Chloe Smith,US,basic,903.555385,13
1,100,Kevin Hall,IN,basic,830.007000,10
2,293,Marcus Perez,US,premium,715.083889,18
3,423,Alice Smith,IN,basic,634.511111,27
4,445,Alice Wilson,UK,basic,615.314211,19
5,136,Ivan Smith,US,basic,615.006842,19
6,313,Samuel Miller,AU,premium,550.470000,9
7,426,Dev Lewis,US,basic,524.316250,24
8,137,Ethan Hall,IN,basic,516.284545,22


In [71]:
checker.p4(solution_4)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 5: Country-Level Summary

Build a country-level summary showing the number of distinct users, accounts, transactions, and total completed transaction amount per country.

<details>
<summary>Hint</summary>

Start from `users`, LEFT JOIN `accounts`, LEFT JOIN `transactions`. Use `COUNT(DISTINCT ...)` for users and accounts. Use `CASE WHEN status = 'completed' THEN amount ELSE 0 END` inside SUM for the amount. Sort by `user_count DESC`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| country | string | |
| user_count | bigint | distinct users |
| account_count | bigint | distinct accounts |
| txn_count | bigint | distinct transactions |
| total_completed_amount | double | 2 decimal places, completed transactions only |

Expected: **5 rows** (one per country: US, UK, IN, SG, AU).

In [83]:
%%solution 5

SELECT
users.country,
COUNT(DISTINCT(acc.user_id)) AS user_count,
COUNT(DISTINCT(acc.account_id)) AS account_count,
COUNT(DISTINCT(txn.txn_id)) AS txn_count,
ROUND(SUM(
    CASE WHEN txn.status = 'completed' THEN txn.amount ELSE 0 END
), 2) AS total_completed_amount
FROM transactions txn
JOIN accounts acc
ON txn.account_id = acc.account_id
JOIN users
ON acc.user_id = users.user_id
GROUP BY users.country
ORDER BY user_count DESC

,country,user_count,account_count,txn_count,total_completed_amount
0,US,206,251,7919,1084575.55
1,UK,97,116,3612,469035.28
2,IN,93,111,2985,458762.38
3,AU,53,65,3022,390831.68
4,SG,51,57,2462,326943.35


In [84]:
checker.p5(solution_5)  # type: ignore[name-defined]  # noqa: F821

True